# M2 — two crossed AODs: arrays, diagonal transport, IM3 ghosts

**What this notebook shows.** Channels `Ax` and `Ay` of the 3D-AODL driven together — i.e. the
conventional 2D-AOD every tweezer lab already owns. Adding the second, crossed deflector buys
three things at once, and each is checked against its closed form:

| # | Physics | Prediction (paper Table I / `docs/conventions.md`) |
|---|---------|------------------------------|
| 1 | **Arrays** — a tone ladder per axis is a grid of tweezers | pitch $=\dfrac{\lambda F}{v}\Delta f = 10.3\ \mu$m per MHz |
| 2 | **Diagonal transport** — chirping *both* axes alike adds two cylinders into a **sphere** | $\bar Z = \dfrac{\lambda F^2}{v^2}\dot f(t_c)$, $\;\Delta F = 0$ |
| 3 | **Intermodulation** — $e^{iCV}$ past first order puts ghosts at $f_j + f_k - f_i$ | ghost amplitude $-\tfrac{i}{8}m^3$ per index triple, suppressed by Schroeder phases |

Point 2 is the M2 headline and the direct answer to M1. One AOD *cannot* move a tweezer without
astigmatising it (notebook 01: $\sigma_{\rm astig}$ reached $-2.8$). Two crossed AODs, chirped
equally, cancel the astigmatism against each other — the spot stays round — but they cannot
cancel the *defocus*: the array leaves the focal plane along $\bar Z$ whenever it moves. Fixing
that needs the counter-propagating partners `Bx`, `By` of M3.

Point 3 is the price of a multi-tone drive. The crystal is not linear: expanding
$e^{iCV}$ past $1 + iCV$ mixes the tones, and the third-order products land back *in band*
(Eqs. S20–S22). They show up twice — as ghost tweezers outside the array, and as a per-trap
intensity error inside it — and the tone phases decide how big both are.

Everything runs through the package's ordinary front door. Physics reference: arXiv:2510.11451
(equations `S#` refer to its Supplement).


In [ ]:
from dataclasses import replace
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

from aodl import (
    ChannelWaveform,
    FrameGrid,
    ToneTrack,
    WaveformSet,
    add_common_ramp,
    array_tones,
    default_1030,
    ramps,
    render_movie,
    schroeder_phases,
    simulate,
)
from aodl.device.aod import channel_lines
from aodl.field.focal import spot_params
from aodl.units import MHz, um, us
from aodl.viz.style import composite, z_color

P = default_1030()                 # paper hardware at lambda = 1030 nm (docs/PLAN.md 1.5)
optics = P.optics
tau = P.channels["Ax"].transit_time
OUT = Path("outputs")              # examples/outputs/ - gitignored


def with_order(params, order):
    """The same hardware with a different weak-drive expansion order (params.py)."""
    return replace(params, channels={name: replace(aod, mixing_order=order)
                                     for name, aod in params.channels.items()})


print(f"deflection scale  = {P.deflection_scale * MHz / um:6.2f} um per MHz     (lambda F / v)")
print(f"lens scale        = {P.lens_scale * (MHz / 1e-3) / um:6.3f} um per MHz/ms  (lambda F^2 / v^2)")
print(f"waist w0          = {optics.waist0 / um:6.3f} um,  Rayleigh range z_R = {optics.rayleigh / um:.3f} um")
print(f"drive strength C  = {P.channels['Ax'].drive_strength:6.2f} rad  -> m = C A = 0.30 at full amplitude")
print(f"mixing order      = {P.channels['Ax'].mixing_order:6d}      (3 = compression + IM3, Eqs. S20-S22)")


## 1. A tone ladder per axis is an array

Stacking AODs *multiplies* their pupils (Eq. S7), so a drive with $M_x$ tones on `Ax` and $M_y$
tones on `Ay` produces every combination: $M_x M_y$ beams, one per tone pair. With the Eq. S18
ladder

$$f^{(n)} = f_{\rm centre} + \Big(n - \tfrac{M-1}{2}\Big)\Delta f, \qquad n = 0 \ldots M-1$$

on each channel, Table I puts the tweezers on a rectangular grid: `Ax` has $s = -1$ and nothing
drives `Bx`, so $X = \tfrac{\lambda F}{v}(f_{Bx} - f_{Ax}) = -\tfrac{\lambda F}{v} f_{Ax}$, and
likewise for $Y$. The pitch is $\tfrac{\lambda F}{v}\Delta f$ — **10.3 µm per MHz** at this
hardware.

`array_tones` builds that ladder, with **Schroeder phases** (Eq. S23/S28,
$\varphi_n = \mathrm{mod}(2\pi n(n-1)/2M,\,2\pi)$) by default: they stop the $M$ tones cresting
together, which is what keeps the intermodulation of §3 down.


In [ ]:
M, DF = 5, 1.0 * MHz
span, t_full = 10 * tau, 2 * tau                       # aperture fully filled at t = 2 tau
ladder = array_tones(M, DF, t1=span)                   # Schroeder phases by default
# the geometry is a first-order statement, so this section uses the linear model and
# section 1.2 below switches the physical mixing_order = 3 back on
square = WaveformSet({"Ax": ladder, "Ay": ladder}, with_order(P, 1))

print("Schroeder phases [rad]:", np.round(schroeder_phases(M), 4))
print("ladder detunings [MHz]:", np.round([tone.f(0.0) / MHz for tone in ladder.tones], 3))

result = simulate(square, [t_full])
terms = result.terms(0)
x_spot = terms.theta1[0] * optics.focal_length / optics.k     # Eq. S11: X = theta1 F / k
y_spot = terms.theta1[1] * optics.focal_length / optics.k
pitch_x = np.diff(np.unique(np.round(x_spot, 12)))

print(f"\nterms (Eq. S7 product)   = {terms.n_terms} = {M} x {M}   (mixing_order = 1)")
print(f"measured pitch           = {pitch_x.mean() / um:.4f} um   predicted {P.deflection_scale * DF / um:.4f} um")
assert np.allclose(pitch_x, P.deflection_scale * DF, rtol=1e-9)


In [ ]:
half = 0.5 * (M + 1) * P.deflection_scale * DF
grid = FrameGrid(-half, half, 481, -half, half, 481)
frame = result.frame(0, grid)

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.imshow(
    composite([(frame, z_color(0.0, 1 * um))], frame.max()),
    extent=[c / um for c in grid.extent],
    origin="lower",
)
for x in np.unique(np.round(x_spot, 12)):
    ax.axvline(x / um, color="#3a7bd5", lw=0.4, alpha=0.35)
for y in np.unique(np.round(y_spot, 12)):
    ax.axhline(y / um, color="#3a7bd5", lw=0.4, alpha=0.35)
ax.set(xlabel="X [µm]", ylabel="Y [µm]", xlim=(grid.x0 / um, grid.x1 / um),
       ylim=(grid.y0 / um, grid.y1 / um),
       title=f"{M}x{M} array, $\\Delta f$ = 1 MHz on Ax and Ay")
ax.text(0.03, 0.03, "lines: Table I grid", transform=ax.transAxes, color="w", fontsize=8)
plt.show()

peak = np.unravel_index(int(np.argmax(frame)), frame.shape)
print(f"brightest pixel at ({grid.x[peak[1]] / um:+.2f}, {grid.y[peak[0]] / um:+.2f}) um "
      f"- nearest grid node ({min(x_spot, key=lambda v: abs(v - grid.x[peak[1]])) / um:+.2f}, "
      f"{min(y_spot, key=lambda v: abs(v - grid.y[peak[0]])) / um:+.2f}) um")


### The catch: equal $\Delta f$ on both axes makes the anti-diagonals coherent

A term's optical frequency is the sum of its tones, $\delta f = f_{Ax} + f_{Ay}$
(`docs/conventions.md` §4), and terms that share one interfere. Two ladders of the *same*
spacing give $\delta f = (n + m)\Delta f$, which depends only on $n + m$ — so every trap on an
anti-diagonal of the array is exactly frequency-degenerate with the others, and `simulate` reports **9 groups, not 25**. Physically that is right (the
traps really are mutually coherent; they simply do not overlap, so nothing interferes visibly),
but it means the per-group metrics are anti-diagonal centroids rather than per-trap numbers.

Detuning one axis fixes it: $\Delta f_y = 1.3$ MHz makes all 25 sums distinct, and each trap gets
its own group. That is also what a lab does when it wants the traps mutually incoherent.


In [ ]:
DFY = 1.3 * MHz
detuned = WaveformSet({"Ax": ladder, "Ay": array_tones(M, DFY, t1=span)}, with_order(P, 1))
print(f"equal spacings  -> {len(result.metrics[0]):3d} groups for {terms.n_terms} terms")
print(f"detuned rows    -> {len(simulate(detuned, [t_full]).metrics[0]):3d} groups")


### Per-trap uniformity, and where the pattern comes from

At `mixing_order=1` every trap carries exactly the same amplitude: the pupil term is a plain
product of two line amplitudes, all equal, and the Eq. S5 aperture polynomial is $(1, 0, 0)$
for the constant envelopes used here — so a *flat* array, to machine precision.

Switch on the physical `mixing_order=3` and a pattern appears. It is not the $\alpha$ product
(that is still $(1,0,0)$); it is intermodulation: an equally spaced ladder is closed under
$f_j + f_k - f_i$, so third-order light lands back **on** the fundamentals and adds coherently
to them. How many index triples reach a given tone depends on where that tone sits in the
ladder, so the centre and the edges are corrected differently — an edge-vs-centre intensity
modulation of a few percent, on top of the overall compression.


In [ ]:
linear = simulate(detuned, [t_full]).spot_table()
mixed = simulate(WaveformSet(detuned.channels, P), [t_full]).spot_table()   # mixing_order = 3
reference = float(linear["power"].mean())
print(f"order 1: per-trap intensity spread (std/mean) = {linear['power'].std() / reference:.2e}")

grid_x = -P.deflection_scale * (np.arange(M) - (M - 1) / 2) * DF
grid_y = -P.deflection_scale * (np.arange(M) - (M - 1) / 2) * DFY
trap = np.full((M, M), np.nan)
for X, Y, power in zip(mixed["x"], mixed["y"], mixed["power"]):
    i, j = int(np.argmin(abs(grid_y - Y))), int(np.argmin(abs(grid_x - X)))
    if abs(grid_y[i] - Y) < 0.01 * optics.waist0 and abs(grid_x[j] - X) < 0.01 * optics.waist0:
        trap[i, j] = power / reference

print(f"\norder 3: trap intensity / first-order intensity (rows = Y, columns = X)")
print(np.round(trap, 4))
print(f"\nedge-vs-centre spread = {np.nanstd(trap) / np.nanmean(trap):.4f}"
      f"   overall compression = {np.nanmean(trap):.4f}")
print(f"light outside the array (ghosts) = {(mixed['power'].sum() - np.nansum(trap) * reference) / (np.nansum(trap) * reference):.2e} of the array")


## 2. Diagonal transport: two cylinders make a sphere

Chirp `Ax` and `Ay` with the *same* minimum-jerk ramp, $0 \to 4$ MHz in 120 µs (Eq. S14). Each
channel contributes a cylindrical lens of the same power (`docs/conventions.md` §3:
$\theta_2 = -\pi\dot f/v^2$, independent of the sound direction), one acting on $x$ and one on
$y$. Table I then gives, with only the A channels driven,

$$\bar Z = \tfrac{1}{2}\frac{\lambda F^2}{v^2}\big(\dot f_{Ax} + \dot f_{Ay}\big)
        = \frac{\lambda F^2}{v^2}\,\dot f(t_c), \qquad
\Delta F = \frac{\lambda F^2}{v^2}\big(\dot f_{Ax} - \dot f_{Ay}\big) = 0 .$$

**Spherical defocus, zero astigmatism** — the M2 acceptance of `docs/PLAN.md` §3. The tweezer
travels along the diagonal *and* out of the focal plane, staying round the whole way; it comes
back to $Z = 0$ only because min-jerk ends at $\dot f = 0$.

The control is the M1 experiment: chirp `Ax` alone and $\Delta F = \tfrac{\lambda F^2}{v^2}\dot f$
comes straight back. Both are plotted together below. Everything is evaluated at the retarded
time $t_c = t - \tau/2$ (`docs/conventions.md` §7).


In [ ]:
SPAN, TIME = 4.0 * MHz, 120.0 * us
chirp = ramps.min_jerk(0.0, TIME, 0.0, SPAN)            # Eq. S14
fdot = chirp.derivative()
one_tone = ChannelWaveform((ToneTrack(freq=chirp),))
frames = np.linspace(0.0, TIME + tau, 121)
t_c = frames - 0.5 * tau                                # docs/conventions.md 7


def run(channels):
    wfs = WaveformSet({name: one_tone for name in channels}, P).with_hold_until(TIME + tau)
    return simulate(wfs, frames)


diagonal, cylinder = run(("Ax", "Ay")), run(("Ax",))
diag, cyl = diagonal.spot_table(), cylinder.spot_table()

zbar_pred = P.lens_scale * fdot(t_c)                    # Table I
print(f"max |Zbar - lens_scale fdot(t_c)| = {np.max(np.abs(diag['z_lab'] - zbar_pred)) / optics.rayleigh:.2e} z_R")
print(f"max |Delta F|, diagonal           = {np.max(np.abs(diag['delta_f'])) / optics.rayleigh:.2e} z_R  -> spherical")
print(f"max |Delta F| - prediction, single = {np.max(np.abs(cyl['delta_f'] - zbar_pred)) / optics.rayleigh:.2e} z_R")
print(f"peak Zbar                         = {np.max(np.abs(diag['z_lab'])) / um:.3f} um = {np.max(np.abs(diag['z_lab'])) / optics.rayleigh:.2f} z_R")
print(f"peak |sigma_astig|: diagonal {np.max(np.abs(diag['sigma_astig'])):.2e}   single axis {np.max(np.abs(cyl['sigma_astig'])):.3f}")
assert np.max(np.abs(diag["delta_f"])) < 1e-12 * optics.rayleigh
assert np.max(np.abs(diag["z_lab"] - zbar_pred)) < 1e-9 * np.max(np.abs(zbar_pred))


In [ ]:
# 1/e^2 radii in the *lab focal plane* Z = 0, where a fixed camera would sit
radii = np.array([
    [np.ravel(spot_params(res.terms(i), optics, 0.0)[2:]) for i in range(len(frames))]
    for res in (diagonal, cylinder)
])

fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.4))
(ax_z, ax_s), (ax_w, ax_t) = axes

ax_z.plot(frames / us, zbar_pred / um, color="k", lw=3, alpha=0.25, label=r"$\lambda F^2 \dot f(t_c)/v^2$")
ax_z.plot(frames / us, diag["z_lab"] / um, color="#3a7bd5", lw=1.4, label=r"diagonal $\bar Z$")
ax_z.plot(frames / us, cyl["z_lab"] / um, color="#c1121f", lw=1.2, ls="--", label=r"single axis $\bar Z$")
ax_z.axhline(optics.rayleigh / um, color="k", lw=0.6, ls=":")
ax_z.set(xlabel="t [µs]", ylabel="Z lab [µm]",
         title="both axes chirped: the focus moves by the full Table I amount")
ax_z.legend(fontsize=8)

ax_s.plot(frames / us, diag["sigma_astig"], color="#3a7bd5", lw=1.6, label="diagonal (Ax + Ay)")
ax_s.plot(frames / us, cyl["sigma_astig"], color="#c1121f", lw=1.6, label="single axis (Ax)")
ax_s.axhline(0.0, color="k", lw=0.8)
ax_s.set(xlabel="t [µs]", ylabel=r"$\sigma_{astig} = \Delta F / z_R$",
         title="the astigmatism cancels exactly (blue: flat zero)")
ax_s.legend(fontsize=8)

# only two curves are distinct: x is chirped in both runs, so w_x is common, and the
# diagonal run has w_y = w_x exactly while the single-axis run leaves y in focus.
ax_w.plot(frames / us, radii[0, :, 0] / um, color="#3a7bd5", lw=3.4, alpha=0.35,
          label=r"diagonal: $w_x = w_y$ (round)")
ax_w.plot(frames / us, radii[1, :, 0] / um, color="#c1121f", lw=1.3, label=r"single axis: $w_x$")
ax_w.plot(frames / us, radii[1, :, 1] / um, color="#c1121f", lw=1.3, ls="--",
          label=r"single axis: $w_y$ (still in focus)")
ax_w.axhline(optics.waist0 / um, color="k", ls=":", lw=0.8)
ax_w.set(xlabel="t [µs]", ylabel="1/e² radius at Z = 0 [µm]",
         title="round but swollen vs stretched")
ax_w.legend(fontsize=8)

ax_t.plot(diag["x"] / um, diag["y"] / um, color="#3a7bd5", lw=2, label="diagonal")
ax_t.plot(cyl["x"] / um, cyl["y"] / um, color="#c1121f", lw=2, label="single axis")
ax_t.set(xlabel="X [µm]", ylabel="Y [µm]", title="path in the image plane")
ax_t.set_aspect("equal")
ax_t.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 3. IM3 ghosts, and what the phases are for

The transmission of the crystal is $e^{iCV}$, not $1 + iCV$. Expanding it (Eqs. S20–S22) keeps,
inside the $+1$ diffraction band, two extra families beyond the fundamentals:

| line | detuning | complex amplitude |
|------|----------|-------------------|
| IM3, three distinct tones | $f_j + f_k - f_i$ | $-\tfrac{i}{8} m_i m_j m_k \, e^{-i(\varphi_j + \varphi_k - \varphi_i)}$ |
| IM3, one tone twice | $2f_j - f_i$ | $-\tfrac{i}{16} m_i m_j^2 \, e^{-i(2\varphi_j - \varphi_i)}$ |

with $m_n = C A_n$ the per-tone modulation depth ($m = 0.3$ here). An equally spaced ladder is
*closed* under both signatures, so the products land either back on a fundamental (a per-trap
intensity error, §1) or on the ladder grid **just outside the array** — ghost tweezers, at
$\pm(\tfrac{M}{2} + 1)\Delta f$, $\pm(\tfrac{M}{2} + 2)\Delta f$, …

Several index sets reach the same ghost, and their phase factors decide whether they add or
cancel. With all $\varphi_n = 0$ they add in step and the prediction is arithmetic,

$$I_{\rm ghost}/I_0 = \Big[\big(N_3\,\tfrac{m^3}{8} + N_2\,\tfrac{m^3}{16}\big)\Big/\tfrac{m}{2}\Big]^2 ,$$

where $N_3, N_2$ count the contributing index sets and $I_0$ is the first-order trap intensity
(the `mixing_order=1` run of the same drive). Schroeder phases scatter the same contributions,
and the ghosts collapse. Below: an 8-tone ladder on `Ax`, $\Delta f = 1$ MHz.


In [ ]:
LADDER_M = 8
detunings = (np.arange(LADDER_M) - (LADDER_M - 1) / 2) * DF
modes = {
    "schroeder": array_tones(LADDER_M, DF, phases="schroeder", t1=span),
    "zero": array_tones(LADDER_M, DF, phases="zero", t1=span),
    "random": array_tones(LADDER_M, DF, phases="random", t1=span, rng=np.random.default_rng(20260901)),
}
tables = {name: simulate(WaveformSet({"Ax": cw}, P), [t_full]).spot_table()
          for name, cw in modes.items()}
I0 = float(simulate(WaveformSet({"Ax": modes["zero"]}, with_order(P, 1)), [t_full])
           .spot_table()["power"].mean())

def split(table):
    """Boolean mask of the programmed traps (the rest of the groups are ghosts)."""
    return np.min(np.abs(table["df_opt"][:, None] - detunings[None, :]), axis=1) < 1.0

spread = {}
print("per-trap intensity spread over the 8 programmed traps, and total ghost light:")
print(f"{'phases':<12}{'std/mean':>10}{'min/max':>10}{'ghosts/traps':>15}")
for name, table in tables.items():
    trap = table["power"][split(table)]
    spread[name] = float(trap.std() / trap.mean())
    ghost = table["power"][~split(table)].sum() / trap.sum()
    print(f"{name:<12}{spread[name]:>10.4f}{trap.min() / trap.max():>10.4f}{ghost:>15.3e}")
assert spread["schroeder"] < 0.25 * spread["zero"]
assert spread["schroeder"] < 0.5 * spread["random"]


In [ ]:
# the furthest IM3 line of a ladder is 2 f_max - f_min = 3 f_max (Eqs. S20-S22)
reach = 1.1 * P.deflection_scale * 3.0 * detunings.max()
wide = FrameGrid(-reach, reach, 1601, -3.0 * um, 3.0 * um, 25)
panels = {name: simulate(WaveformSet({"Ax": cw}, P), [t_full]).frame(0, wide)
          for name, cw in modes.items() if name != "random"}
peak = max(float(f.max()) for f in panels.values())

fig, axes = plt.subplots(3, 1, figsize=(11.0, 5.6), sharex=True,
                         gridspec_kw={"height_ratios": [1, 1, 2.2]})
for ax, (name, panel) in zip(axes, panels.items()):
    ax.imshow(np.maximum(panel / peak, 1e-9), extent=[c / um for c in wide.extent], origin="lower",
              aspect="auto", cmap="inferno", norm=LogNorm(vmin=1e-6, vmax=1.0))
    ax.set(ylabel="Y [µm]", title=f"{name} phases (log intensity)")
for name, color in (("schroeder", "#3a7bd5"), ("zero", "#c1121f")):
    axes[2].semilogy(wide.x / um, np.maximum(panels[name][wide.ny // 2] / peak, 1e-10),
                     color=color, lw=1.1, label=f"{name} phases")
for f in detunings:
    axes[2].axvline(-P.deflection_scale * f / um, color="#8ac926", lw=0.6, alpha=0.5)
axes[2].set(xlabel="X [µm]", ylabel="I / I$_{max}$", ylim=(1e-7, 3.0),
            title="cut through Y = 0 (green: the 8 programmed traps)")
axes[2].legend(fontsize=8, loc="lower right", framealpha=0.85)
plt.tight_layout()
plt.show()


In [ ]:
def im3_paths(target):
    """(N3, N2): index sets with f_j + f_k - f_i = target and with 2 f_j - f_i = target."""
    n = len(detunings)
    n3 = sum(abs(detunings[j] + detunings[k] - detunings[i] - target) < 1.0
             for j, k in combinations(range(n), 2) for i in range(n) if i not in (j, k))
    n2 = sum(abs(2 * detunings[j] - detunings[i] - target) < 1.0
             for j in range(n) for i in range(n) if i != j)
    return int(n3), int(n2)


def intensity_at(table, f):
    """Group intensity at optical frequency ``f``, relative to a first-order trap."""
    return float(table["power"][int(np.argmin(np.abs(table["df_opt"] - f)))]) / I0


m = P.channels["Ax"].drive_strength
ghosts = tables["zero"]["df_opt"][~split(tables["zero"])]
ghosts = ghosts[np.argsort([-intensity_at(tables["zero"], f) for f in ghosts])]

print(f"top IM3 ghosts of the {LADDER_M}-tone ladder, m = {m}, relative to a first-order trap")
print(f"{'df [MHz]':>9}{'X [um]':>9}{'N3':>4}{'N2':>4}{'aligned pred':>14}"
      f"{'zero':>12}{'random':>12}{'schroeder':>12}{'suppressed':>12}")
for f in ghosts[:8]:
    n3, n2 = im3_paths(f)
    predicted = ((n3 * m**3 / 8 + n2 * m**3 / 16) / (0.5 * m)) ** 2
    measured = {name: intensity_at(table, f) for name, table in tables.items()}
    print(f"{f / MHz:>9.2f}{-P.deflection_scale * f / um:>9.2f}{n3:>4d}{n2:>4d}{predicted:>14.3e}"
          f"{measured['zero']:>12.3e}{measured['random']:>12.3e}{measured['schroeder']:>12.3e}"
          f"{measured['zero'] / measured['schroeder']:>11.0f}x")
    assert abs(measured["zero"] / predicted - 1.0) < 1e-3      # the -(i/8) m^3 class, exactly


## 4. The movie: a 3×3 array in diagonal flight

A 3×3 array (`array_tones` on both channels) plus the *same* min-jerk chirp added to every tone
— `add_common_ramp`, the lateral term of Eq. S19 — so the whole array translates rigidly along
the diagonal while its spacing stays put.

The view is the package default, `mode="tracked"`: the XY plane follows the scene's
power-weighted best focus. Notebook 01 deliberately used `mode="fixed"` instead, because for a
*single* AOD the tracked plane sits half way between two line foci, where the spot is round but
swollen — the circle of least confusion, which hides the very astigmatism M1 was about. Here
there is no astigmatism to hide: both axes are chirped equally, the defocus is **spherical**, and
the tracked plane is a real focus. So the array stays sharp all the way through, and the only
sign of the excursion is the **hue** — white in the focal plane, red as $\bar Z$ climbs to
$\approx 1.9\,z_R$ mid-move and back. The XZ panel on the right shows the same climb directly,
with the tracked plane marked.


In [ ]:
from IPython.display import Video

moving = add_common_ramp(array_tones(3, DF, t0=0.0, t1=TIME), chirp)
array_flight = WaveformSet({"Ax": moving, "Ay": moving}, P).with_hold_until(TIME + tau)
flight = simulate(array_flight, np.linspace(0.0, TIME + tau, 48))

print(f"{flight.terms(-1).n_terms} pupil terms in {len(flight.metrics[-1])} frequency groups "
      f"(3x3 traps + IM3 ghosts, mixing_order=3)")
print(f"tracked plane sweeps 0 -> {flight.tracked_z().max() / um:.2f} um "
      f"= {flight.tracked_z().max() / optics.rayleigh:.2f} z_R and back")

movie = render_movie(flight, OUT / "02_diagonal.mp4", mode="tracked", fps=20, spectrogram_panel=True)
print(f"{movie}  ({movie.stat().st_size / 1e3:.0f} kB, {flight.n_frames} frames)")
Video(str(movie), embed=True, html_attributes="controls loop")


## What M3 adds

Two crossed AODs give arrays and in-plane motion, but the numbers above show what they cannot
do: **every move is a defocus**. $\bar Z = \tfrac{\lambda F^2}{v^2}\dot f$ is not optional — it
is the same chirp that moves the tweezer — so the array of §4 climbs 1.9 Rayleigh ranges out of
the focal plane simply because it is travelling. Cancelling the astigmatism (§2) fixed the
*shape* of the spot, not its depth.

The counter-propagating partners `Bx` and `By` are what fix it. Their sound runs the other way,
so their deflection *subtracts* while their lensing still *adds* (`docs/conventions.md` §3:
$\theta_1 \propto s f$ carries the sound sign, $\theta_2 \propto \dot f$ does not). That splits
the four channels into independent knobs — the Eq. S19 synthesis of M3:

* **counter-chirp within a pair** ($\dot f_{Ax} = -\dot f_{Bx}$) → in-plane motion with *zero*
  focal shift: the paper's key result, and the thing this notebook cannot do;
* **co-chirp all four** → pure $\bar Z$ motion, laterally static and round;
* **$\Delta F = 0$ as a constraint** → astigmatism-free 3D control, three degrees of freedom for
  $(X, Y, Z)$, all the way to the 10×10 lift-traverse-lower user story.

M4 then adds fading-Shepard ladders (Eqs. S24–S28) for the sustained $Z$ offsets that the ±10 MHz
band cannot hold with a single chirp — and the Schroeder phases of §3 come back there (Eq. S28)
for exactly the same reason.
